In [ ]:
# 4_Final_Inference.ipynb

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import random
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K

# --- ROBUST IMPORT FOR KERAS SERIALIZABLE DECORATOR ---
# Handles the namespace changes between TF 2.12, TF 2.15+, and Keras 3
try:
    from keras.saving import register_keras_serializable
except ImportError:
    from tensorflow.keras.utils import register_keras_serializable

# --- 1. CONFIGURATION ---

# Define available models
available_models = {
    "1": ("UNet (Standard)",
          "../saved_models/unet_oil_spill.keras"),

    "2": ("DeepLabV3+ (Experimental)",
          "../saved_models/deeplabv3_oil_spill.h5"),

    "3": ("Sparse Architecture (Advanced)",
          "../saved_models/sparse_unet_oil_spill.keras")
}

print("\n🤖 SELECT AI MODEL:")
print("1. UNet (Standard)")
print("2. DeepLabV3+ (Experimental)")
print("3. Sparse Architecture (Advanced)")

choice = input("\n👉 Enter model number (1, 2 or 3): ")

if choice in available_models:
    MODEL_NAME, MODEL_PATH = available_models[choice]
else:
    MODEL_NAME, MODEL_PATH = available_models["1"]

print(f"\n✅ Configuration Locked: Using {MODEL_NAME}")
print(f"📂 Path: {MODEL_PATH}")

AIS_DATA_PATH = '../data/ais_data/vessel_data_clean.csv' 

# AUTOMATICALLY FIND A RANDOM TEST IMAGE
TEST_DIR = '../data/test/images' 
TEST_IMG_PATH = None

if os.path.exists(TEST_DIR):
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    files = os.listdir(TEST_DIR)
    images = [f for f in files if os.path.splitext(f)[1].lower() in valid_extensions]

    if len(images) > 0:
        selected_file = random.choice(images)
        TEST_IMG_PATH = os.path.join(TEST_DIR, selected_file)
        print(f"🎲 Randomly selected test image: {selected_file}")
    else:
        print(f"WARNING: No valid images found in '{TEST_DIR}'.")
        TEST_IMG_PATH = 'dummy_path.jpg'
else:
    print(f"Error: Directory {TEST_DIR} does not exist.")
    TEST_IMG_PATH = 'dummy_path.jpg' 

# --- 2. LOAD MODEL ---
print("Loading Model...")

# Define ALL custom functions across UNet, DeepLab, and Sparse UNet
# Registered explicitly so Keras 3 /.keras format knows how to rebuild them

@register_keras_serializable(name='dice_coef')
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

@register_keras_serializable(name='dice_loss')
def dice_loss(y_true, y_pred, smooth=1e-6):
    return 1.0 - dice_coef(y_true, y_pred, smooth)

@register_keras_serializable(name='dice_coeff_metric')
def dice_coeff_metric(y_true, y_pred, smooth=1e-6):
    y_pred_metric = K.cast(K.greater(y_pred, 0.5), K.floatx())
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred_metric)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

@register_keras_serializable(name='iou_metric')
def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred_metric = K.cast(K.greater(y_pred, 0.5), K.floatx())
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred_metric)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

@register_keras_serializable(name='focal_loss')
def focal_loss(y_true, y_pred, alpha=0.75, gamma=2.0):
    y_true = tf.cast(y_true, tf.float32)
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    cross_entropy = -y_true * K.log(y_pred)
    weight = alpha * K.pow(1.0 - y_pred, gamma)
    loss = weight * cross_entropy
    return K.mean(K.sum(loss, axis=-1))

@register_keras_serializable(name='sparse_focal_dice_loss')
def sparse_focal_dice_loss(y_true, y_pred):
    return 0.5 * focal_loss(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

@register_keras_serializable(name='dice_coeff_soft')
def dice_coeff_soft(y_true, y_pred, smooth=1e-6):
    return dice_coef(y_true, y_pred, smooth)

@register_keras_serializable(name='dice_metric_hard')
def dice_metric_hard(y_true, y_pred, smooth=1e-6):
    return dice_coeff_metric(y_true, y_pred, smooth)

# Combine into custom objects dictionary just in case
custom_objects = {
    'dice_coef': dice_coef,
    'dice_loss': dice_loss,
    'dice_coeff_metric': dice_coeff_metric,
    'iou_metric': iou_metric,
    'focal_loss': focal_loss,
    'sparse_focal_dice_loss': sparse_focal_dice_loss,
    'dice_coeff_soft': dice_coeff_soft,
    'dice_metric_hard': dice_metric_hard
}

try:
    model = load_model(MODEL_PATH, custom_objects=custom_objects)
    print("✅ Full model loaded successfully!")
except Exception as e:
    model = None
    print(f"❌ Error loading model: {e}")

# --- 3. PREDICT FUNCTION ---
def predict_spill(image_path, threshold=0.6):
    """
    Runs inference on a processed SAR image tensor.
    Threshold set to 0.6 to counter Focal Loss confidence mapping
    and reduce false positives from SAR speckle.
    """
    if model is None:
        return np.zeros((256,256,3)), np.zeros((256,256,1)), np.zeros((256,256,1))

    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}")
        return np.zeros((256,256,3)), np.zeros((256,256,1)), np.zeros((256,256,1)) 

    original_img = cv2.imread(image_path)
    
    # Safety check for corrupted or unreadable images
    if original_img is None:
        print(f"Error: Unable to read image file {image_path}. Check file format.")
        return np.zeros((256,256,3)), np.zeros((256,256,1)), np.zeros((256,256,1))
    
    # 1. SMART PREPROCESSING: Prevents shape errors regardless of model choice
    expected_channels = model.input_shape[-1]
    
    if expected_channels == 1:
        proc_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2GRAY)
        proc_img = cv2.resize(proc_img, (256, 256))
        img_input = np.expand_dims(proc_img, axis=-1) 
    else:
        proc_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
        proc_img = cv2.resize(proc_img, (256, 256))
        img_input = proc_img
        
    img_input = np.expand_dims(img_input, axis=0) / 255.0 
    
    # 2. VISUALIZATION IMAGE
    vis_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    vis_img = cv2.resize(vis_img, (256, 256))
    
    # 3. PREDICTION
    raw_pred = model.predict(img_input, verbose=0)[0]
    print(f"DEBUG: Model Output Stats -> Max: {np.max(raw_pred):.4f}, Mean: {np.mean(raw_pred):.4f}")

    # Apply dynamic threshold
    mask_pred = (raw_pred > threshold).astype(np.uint8) 
    
    return vis_img, mask_pred, raw_pred

# --- 4. ANOMALY DETECTION ---
def detect_anomaly(ais_csv, spill_lat, spill_lon, search_radius=0.5):
    if not os.path.exists(ais_csv):
        return []
    df = pd.read_csv(ais_csv)
    nearby_ships = df[
        (df['LAT'] > spill_lat - search_radius) & (df['LAT'] < spill_lat + search_radius) & 
        (df['LON'] > spill_lon - search_radius) & (df['LON'] < spill_lon + search_radius)
    ]
    report_data = []
    for _, ship in nearby_ships.iterrows():
        status = "STOPPED (SUSPECT)" if ship['SOG'] < 1.0 else "MOVING"
        report_data.append({
            "Status": status, "Vessel Name": str(ship['VesselName']),
            "MMSI": ship['MMSI'], "Speed (knots)": ship['SOG']
        })
    return report_data

# --- 5. DAMAGE ASSESSMENT ---
def assess_damage(mask, raw_pred):
    oil_pixel_count = np.count_nonzero(mask)
    if oil_pixel_count > 0:
        avg_confidence = np.mean(raw_pred[mask > 0]) * 100
        total_pixels = mask.shape[0] * mask.shape[1]
        spill_percentage = (oil_pixel_count / total_pixels) * 100
    else:
        avg_confidence, spill_percentage = 0.0, 0.0
    
    total_area_km2 = (oil_pixel_count * 100) / 1_000_000
    
    print("\n--- DAMAGE ASSESSMENT REPORT ---")
    print(f"Estimated Spill Area:  {total_area_km2:.4f} sq. km")
    print(f"Average AI Confidence: {avg_confidence:.2f}%")

    if total_area_km2 > 1.0: print("SEVERITY: CRITICAL")
    elif total_area_km2 > 0.1: print("SEVERITY: HIGH")
    elif total_area_km2 > 0.0: print("SEVERITY: MODERATE")
    else: print("SEVERITY: NONE")

# --- 6. RUN PIPELINE ---
if model is None:
    print("FATAL ERROR: Model not defined. Pipeline halted.")
elif TEST_IMG_PATH is None or not os.path.exists(TEST_IMG_PATH):
    print("FATAL ERROR: Invalid test image path.")
else:
    print("\n--- STARTING ANALYSIS ---")
    final_img, final_mask, raw_pred = predict_spill(TEST_IMG_PATH) 

    # Visualization
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1); plt.title("Satellite Input"); plt.imshow(final_img); plt.axis('off')
    
    # Explicit vmin and vmax forces 0 to be black and 255 to be white
    plt.subplot(1, 3, 2); plt.title("AI Mask")
    plt.imshow(final_mask.squeeze() * 255, cmap='gray', vmin=0, vmax=255)
    plt.axis('off')
    
    # Bulletproof overlay logic that prevents background tinting/dimming
    overlay = final_img.copy()
    mask_bool = final_mask.squeeze() == 1  # 2D boolean array of the spill
    
    # Create a solid red canvas
    red_layer = np.zeros_like(final_img)
    red_layer[:, :, 0] = 255 
    
    # Blend the red canvas with the original image
    blended = cv2.addWeighted(overlay, 0.7, red_layer, 0.3, 0)
    
    # Inject the blended pixels ONLY where the mask is True. 
    # Background remains 100% original.
    overlay[mask_bool] = blended[mask_bool]
    
    plt.subplot(1, 3, 3); plt.title("Forensic Overlay"); plt.imshow(overlay); plt.axis('off')
    plt.show()

    assess_damage(final_mask, raw_pred)
    
    anomalies = detect_anomaly(AIS_DATA_PATH, 28.5, -90.5)
    print("\n" + "="*50 + "\n🚢 NEARBY VESSEL ACTIVITY REPORT\n" + "="*50)
    if anomalies:
        print(pd.DataFrame(anomalies).sort_values(by="Speed (knots)").to_string(index=False))
    else:
        print("No vessels detected.")